# CropYield AI — FAOSTAT web scraping to Streamlit
**Statistical Interface Lab | Regression project**

This notebook scrapes the FAOSTAT QCL page for the official download link, downloads the normalized archive, prepares a yield dataset, performs EDA, trains baseline regression models, saves model artifacts, and exports files for the Streamlit GUI.

> FAOSTAT provides country/area-level statistics. This is not a farm-specific prediction model. Yield units are normalized to kg/ha when the source unit is recognized.

## 1. Install packages

In [ ]:
!pip -q install pandas numpy scikit-learn joblib requests beautifulsoup4 plotly matplotlib seaborn

## 2. Create project folders and scrape the FAOSTAT page

In [ ]:
from pathlib import Path
import requests, zipfile, json, warnings
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE = Path('/content/CropYield_AI')
(BASE/'data/raw').mkdir(parents=True, exist_ok=True)
(BASE/'models').mkdir(parents=True, exist_ok=True)
(BASE/'src').mkdir(parents=True, exist_ok=True)

FAOSTAT_PAGE = 'https://www.fao.org/faostat/en/#data/QCL'
FALLBACK_URL = 'https://bulks-faostat.fao.org/production/Production_Crops_Livestock_E_All_Data_(Normalized).zip'
headers = {'User-Agent': 'CropYieldAILabProject/1.0 (educational research)'}

response = requests.get(FAOSTAT_PAGE, headers=headers, timeout=45)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'html.parser')
candidate_links = []
for tag in soup.find_all(['a','link']):
    href = tag.get('href','')
    label = ' '.join([tag.get_text(' ',strip=True), tag.get('title',''), tag.get('aria-label','')]).lower()
    full_url = urljoin(FAOSTAT_PAGE, href)
    if 'production_crops_livestock' in full_url.lower() or ('bulk' in label and 'crop' in label):
        candidate_links.append(full_url)
candidate_links = list(dict.fromkeys(candidate_links))
download_url = next((u for u in candidate_links if u.lower().endswith('.zip')), FALLBACK_URL)
print('FAOSTAT page:', FAOSTAT_PAGE)
print('Scraped candidate links:', candidate_links[:10] or 'No direct archive link in HTML; using official bulk-download fallback')
print('Chosen archive:', download_url)

## 3. Download and read the official normalized archive

In [ ]:
zip_path = BASE/'data/raw/faostat_qcl_normalized.zip'
with requests.get(download_url, headers=headers, stream=True, timeout=(30,180)) as r:
    r.raise_for_status()
    with open(zip_path, 'wb') as f:
        for chunk in r.iter_content(chunk_size=1024*1024):
            if chunk:
                f.write(chunk)
assert zipfile.is_zipfile(zip_path), 'Download is not a valid ZIP; inspect the URL/response.'
with zipfile.ZipFile(zip_path) as zf:
    csvs = [n for n in zf.namelist() if n.lower().endswith('.csv')]
    assert csvs, 'No CSV found in the archive.'
    csv_name = max(csvs, key=lambda n: zf.getinfo(n).file_size)
    with zf.open(csv_name) as f:
        raw = pd.read_csv(f, low_memory=False, encoding_errors='replace')
raw.columns = [str(c).strip() for c in raw.columns]
print('CSV inside ZIP:', csv_name)
print('Raw shape:', raw.shape)
display(raw.head())
print(raw.columns.tolist())

## 4. Prepare yield records

In [ ]:
def find_col(df, candidates):
    lookup = {c.lower().replace('_',' ').strip(): c for c in df.columns}
    for name in candidates:
        key = name.lower().replace('_',' ').strip()
        if key in lookup:
            return lookup[key]
    return None

area_col = find_col(raw, ['Area','Country','Area Name'])
item_col = find_col(raw, ['Item','Item Name','Crop'])
element_col = find_col(raw, ['Element','Element Name'])
year_col = find_col(raw, ['Year','Year Code'])
value_col = find_col(raw, ['Value','Value Numeric'])
unit_col = find_col(raw, ['Unit'])
assert all([area_col,item_col,element_col,year_col,value_col]), f'Required columns not found: {raw.columns.tolist()}'

df = raw.rename(columns={area_col:'Area', item_col:'Item', element_col:'Element', year_col:'Year', value_col:'Value'}).copy()
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
df['Value'] = pd.to_numeric(df['Value'].astype(str).str.replace(',','',regex=False), errors='coerce')
df = df.dropna(subset=['Area','Item','Element','Year','Value'])
df['Year'] = df['Year'].astype(int)
if 'Area Code (M49)' in df.columns:
    df = df[df['Area Code (M49)'].notna()].copy()
df['Element_norm'] = df['Element'].astype(str).str.strip().str.lower()

yield_rows = df[df['Element_norm'].eq('yield')].copy()
area_rows = df[df['Element_norm'].str.contains('area harvested', na=False)].copy()
keys = ['Area','Item','Year']
assert not yield_rows.empty, "No Element='Yield' records found. Inspect available Element values."

target = yield_rows[keys+['Value']].rename(columns={'Value':'Yield_raw'}).groupby(keys,as_index=False)['Yield_raw'].mean()
if not area_rows.empty:
    area_feature = area_rows[keys+['Value']].rename(columns={'Value':'Area_harvested_ha'}).groupby(keys,as_index=False)['Area_harvested_ha'].mean()
    target = target.merge(area_feature,on=keys,how='left')
else:
    target['Area_harvested_ha'] = np.nan

source_unit = 'unknown'
if unit_col:
    units = df[df['Element_norm'].eq('yield')][unit_col].dropna().astype(str)
    if len(units): source_unit = units.mode().iloc[0]
print('Source yield unit:', source_unit)
unit_key = source_unit.lower().replace(' ','')
if unit_key in {'hg/ha','hgperha'}:
    target['Yield_kg_ha'] = target['Yield_raw'] * 0.1
elif unit_key in {'kg/ha','kgperha'}:
    target['Yield_kg_ha'] = target['Yield_raw']
elif unit_key in {'t/ha','tonnes/ha','tonnesperha'}:
    target['Yield_kg_ha'] = target['Yield_raw'] * 1000
else:
    target['Yield_kg_ha'] = target['Yield_raw']
    warnings.warn(f"Unit {source_unit!r} not recognized; output retains source numeric scale. Verify units before reporting.")

data = target.rename(columns={'Area':'Country','Item':'Crop'})
data = data.replace([np.inf,-np.inf],np.nan).dropna(subset=['Yield_kg_ha'])
data = data[(data['Yield_kg_ha']>0) & (data['Yield_kg_ha'] < data['Yield_kg_ha'].quantile(.995))]
data = data.drop_duplicates(subset=['Country','Crop','Year'])
data = data[['Country','Crop','Year','Area_harvested_ha','Yield_kg_ha']]
data.to_csv(BASE/'data/crop_yield_training.csv',index=False)
print('Prepared dataset shape:',data.shape)
display(data.head())
display(data.describe(include='all').T)

## 5. Exploratory data analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(9,4))
sns.histplot(data['Yield_kg_ha'], bins=50, kde=True)
plt.title('Distribution of recorded crop yield')
plt.xlabel('Yield (kg/ha, if source unit recognized)')
plt.tight_layout()
plt.show()

yearly = data.groupby('Year',as_index=False)['Yield_kg_ha'].mean()
plt.figure(figsize=(10,4))
sns.lineplot(data=yearly,x='Year',y='Yield_kg_ha')
plt.title('Mean recorded yield by year')
plt.tight_layout()
plt.show()

top_crops = data.groupby('Crop')['Yield_kg_ha'].mean().sort_values(ascending=False).head(15).sort_values()
plt.figure(figsize=(9,6))
top_crops.plot(kind='barh')
plt.title('Crops with highest mean recorded yield (top 15)')
plt.xlabel('Mean yield')
plt.tight_layout()
plt.show()

print('Rows:',len(data),'Countries/areas:',data['Country'].nunique(),'Crops:',data['Crop'].nunique())

## 6. Train and compare regression models

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import joblib

features = ['Country','Crop','Year','Area_harvested_ha']
target_col = 'Yield_kg_ha'
X, y = data[features].copy(), data[target_col].copy()
preprocess = ColumnTransformer([
    ('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),
                     ('onehot',OneHotEncoder(handle_unknown='ignore'))]),['Country','Crop']),
    ('num',Pipeline([('imputer',SimpleImputer(strategy='median')),
                     ('scale',StandardScaler())]),['Year','Area_harvested_ha'])
])
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=.2,random_state=42)

results, fitted = [], {}
for name, estimator in [
    ('Linear Regression',LinearRegression()),
    ('Random Forest',RandomForestRegressor(n_estimators=250,min_samples_leaf=2,random_state=42,n_jobs=-1))
]:
    pipe = Pipeline([('preprocess',preprocess),('model',estimator)])
    pipe.fit(X_train,y_train)
    pred = pipe.predict(X_test)
    results.append({'Model':name,'R2':r2_score(y_test,pred),
                    'MAE':mean_absolute_error(y_test,pred),
                    'RMSE':np.sqrt(mean_squared_error(y_test,pred))})
    fitted[name] = pipe

metrics = pd.DataFrame(results).sort_values(['RMSE','MAE']).reset_index(drop=True)
display(metrics)
best_name = metrics.iloc[0]['Model']
best_model = fitted[best_name]
joblib.dump(best_model, BASE/'models/best_model.joblib')
metrics.to_csv(BASE/'models/model_metrics.csv',index=False)
model_metadata = {
    'best_model':best_name,
    'features':features,
    'target':target_col,
    'source':'FAOSTAT QCL - Production: Crops and livestock products',
    'source_unit':source_unit,
    'target_unit':'kg/ha when source unit recognized; otherwise source unit scale',
    'training_rows':int(len(data)),
    'selection_metric':'lowest RMSE on random 80/20 split',
    'note':'Production is excluded as an input feature to avoid target leakage.'
}
(BASE/'models/model_metadata.json').write_text(json.dumps(model_metadata,indent=2),encoding='utf-8')
print('Best baseline model:',best_name)

## 7. Actual vs predicted and residual analysis

In [ ]:
pred = best_model.predict(X_test)
plt.figure(figsize=(6,6))
plt.scatter(y_test,pred,alpha=.25)
lims = [min(y_test.min(),pred.min()),max(y_test.max(),pred.max())]
plt.plot(lims,lims,'--')
plt.xlabel('Actual yield')
plt.ylabel('Predicted yield')
plt.title(f'Actual vs predicted — {best_name}')
plt.tight_layout()
plt.show()

residuals = y_test - pred
plt.figure(figsize=(9,4))
plt.scatter(pred,residuals,alpha=.25)
plt.axhline(0,linestyle='--')
plt.xlabel('Predicted yield')
plt.ylabel('Residual (actual - predicted)')
plt.title('Residual plot')
plt.tight_layout()
plt.show()

## 8. Export artifacts for the Streamlit website

In [ ]:
import shutil
shutil.make_archive('/content/CropYield_AI_artifacts','zip',BASE)
print('Artifact ZIP created: /content/CropYield_AI_artifacts.zip')
print('Download it from the Colab Files panel, or run:')
print("from google.colab import files; files.download('/content/CropYield_AI_artifacts.zip')")